# AMEX Enterprise Credit Risk Platform
## Notebook 24 — Phase 1, Problem 2: Risk Tier Classification — Comprehensive Reporting
### Problem Statement 2 of 14: Risk Tier Classification

CRISP-DM stage: **Evaluation / Reporting**. Combines Notebook 17's schema-aware rollup pattern (reading every prior Problem 2 notebook's real summary JSON) with Notebook 14's business/financial-impact narrative style, scoped down to Problem 2's real, measured tier-quality findings. Hard dependency on Notebook 19 (the real policy); Notebooks 20-23 are read opportunistically, exactly like Notebook 17's own resilience pattern -- a notebook that has not yet run is reported as "not yet run", never fabricated.

**What this notebook does, all real:**

- A schema-aware rollup of Notebooks 19-23's real summary artifacts -- policy, model-development KPI compliance, validation statistics, deployment self-test result, and monitoring alert status.
- A condensed, honestly-scoped financial narrative: the real bad-rate separation Notebook 20 measured between the highest- and lowest-risk tier, translated into an illustrative risk-based-pricing benefit estimate under explicit, editable, clearly labeled `ASSUMPTION` dollar figures -- this Kaggle dataset carries no real revenue or cost data for any institution, exactly the same honesty boundary Notebook 14 states for Problem 1.
- A Problem 2 "at a glance" headline panel and a completion tracker across Notebooks 19-25.

**Deliverables:** `risk_tier_rollup.json`, charts, `Risk_Tier_Comprehensive_Report.docx`, `notebook_24_summary.json`.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG & NOTEBOOK 19 (REQUIRED)
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config & Notebook 19 (Required)")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.\nFix: run 01_business_understanding.ipynb first.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (_resource_limits.get("warp_thread_count") or PROJECT_CONFIG.get("warp_thread_count")
                      or PROJECT_CONFIG["hardware"].get("logical_cores_detected"))

_REQUIRED_PILLARS = {
    "risk_tier_policy": "Problem2_Risk_Tier_Classification/01_Risk_Tier_Policy",
    "risk_tier_modeling": "Problem2_Risk_Tier_Classification/02_Risk_Tier_Modeling",
    "risk_tier_validation": "Problem2_Risk_Tier_Classification/03_Risk_Tier_Validation",
    "risk_tier_deployment": "Problem2_Risk_Tier_Classification/04_Risk_Tier_Deployment",
    "risk_tier_monitoring": "Problem2_Risk_Tier_Classification/05_Risk_Tier_Monitoring",
    "risk_tier_reporting": "Problem2_Risk_Tier_Classification/06_Risk_Tier_Reporting",
    "risk_tier_packaging": "Problem2_Risk_Tier_Classification/07_Risk_Tier_Packaging",
}
_config_healed = False
for _key, _rel_path in _REQUIRED_PILLARS.items():
    if _key not in PILLAR_DIRS:
        PILLAR_DIRS[_key] = PROJECT_ROOT / _rel_path
        PROJECT_CONFIG["pillar_dirs"][_key] = str(PILLAR_DIRS[_key])
        _config_healed = True
        print(f"NOTE: '{_key}' was missing from project_config.json -- added automatically as {PILLAR_DIRS[_key]}")
if _config_healed:
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(PROJECT_CONFIG, f, indent=2)
    print("\u2705 project_config.json updated in place -- no need to re-run Notebook 01.")

RISK_TIER_REPORTING_DIR = PILLAR_DIRS["risk_tier_reporting"]
RISK_TIER_REPORTING_DIR.mkdir(parents=True, exist_ok=True)

NB19_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_19_summary.json"
if not NB19_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"{NB19_SUMMARY_PATH} not found.\nNotebook 24 has a hard dependency on Notebook 19 -- "
        f"fix: run 19_risk_tier_business_understanding.ipynb first."
    )
with open(NB19_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB19_SUMMARY = json.load(f)

# --- Notebook 19's summary JSON does not itself carry the champion's holdout
#     metrics (only champion_model/n_tiers/primary_method) -- those live in
#     the real, saved risk_tier_policy.json Notebook 19 also writes. Read it
#     directly rather than assuming a key that is not actually there. ---
RISK_TIER_POLICY_PATH = PILLAR_DIRS["risk_tier_policy"] / "risk_tier_policy.json"
if not RISK_TIER_POLICY_PATH.exists():
    raise FileNotFoundError(f"{RISK_TIER_POLICY_PATH} not found.\nFix: re-run 19_risk_tier_business_understanding.ipynb.")
with open(RISK_TIER_POLICY_PATH, "r", encoding="utf-8") as f:
    RISK_TIER_POLICY = json.load(f)

print(f"Reporting artifacts will be written under: {RISK_TIER_REPORTING_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION, LIBRARY IMPORTS & ADAPTIVE RAM CEILING
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration, Library Imports & Adaptive RAM Ceiling")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

missing = []
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
    from docx.shared import Inches
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    missing.append("python-docx")

if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) + "\n"
                       f"Fix: pip install {' '.join(missing)}")


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()
_live_vm = psutil.virtual_memory()
ADAPTIVE_RAM_FRACTION = _resource_limits.get("ram_fraction_cap", 0.90)
MAX_RAM_BYTES = int(_live_vm.available * ADAPTIVE_RAM_FRACTION)

print(f"Adaptive RAM ceiling (this run) : {MAX_RAM_BYTES / 1e9:.2f} GB")
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: SCHEMA-AWARE ROLLUP -- NOTEBOOKS 19-23 (19 REQUIRED, 20-23 OPPORTUNISTIC)
# =============================================================================
_section("SECTION 3: Schema-Aware Rollup -- Notebooks 19-23")

# --- Same resilience pattern as Notebook 17: a summary that has not been
#     written yet (that notebook has not run) is reported honestly as
#     "not yet run" -- never fabricated, never silently skipped without a
#     trace in the rollup. ---
NOTEBOOK_TITLES = {
    19: "Business Understanding & Policy", 20: "Model Development", 21: "Validation",
    22: "Deployment", 23: "Monitoring",
}
_summaries = {}
_rollup_rows = []
for _n in range(19, 24):
    _p = ARTIFACTS_DIR / f"notebook_{_n}_summary.json"
    if _p.exists():
        with open(_p, "r", encoding="utf-8") as f:
            _summaries[_n] = json.load(f)
        _status = "Complete"
    else:
        _summaries[_n] = None
        _status = "Not Yet Run"
    _rollup_rows.append({"notebook": _n, "title": NOTEBOOK_TITLES[_n], "status": _status,
                          "summary_file": _p.name if _p.exists() else "(missing)"})

rollup_status_df = pd.DataFrame(_rollup_rows)
print(rollup_status_df.to_string(index=False))

NB20_SUMMARY = _summaries.get(20)
NB21_SUMMARY = _summaries.get(21)
NB22_SUMMARY = _summaries.get(22)
NB23_SUMMARY = _summaries.get(23)

_n_complete = int((rollup_status_df["status"] == "Complete").sum())
print(f"\n{_n_complete} of 5 Problem 2 notebooks (19-23) have real summary artifacts on disk.")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: PROBLEM 2 "AT A GLANCE" -- HEADLINE KPI PANEL
# =============================================================================
_section("SECTION 4: Problem 2 At A Glance -- Headline KPI Panel")

AT_A_GLANCE = {
    "champion_model": NB19_SUMMARY["champion_model"],
    "champion_holdout_auc": RISK_TIER_POLICY.get("champion_holdout_auc"),
    "champion_holdout_amex_metric": RISK_TIER_POLICY.get("champion_holdout_amex_metric"),
    "n_tiers": NB19_SUMMARY["n_tiers"],
    "primary_method": NB19_SUMMARY["primary_method"],
}
if NB20_SUMMARY:
    AT_A_GLANCE["primary_method_full_kpi_pass"] = NB20_SUMMARY.get("primary_method_full_kpi_pass")
    AT_A_GLANCE["split_half_psi"] = NB20_SUMMARY.get("split_half_psi")
    AT_A_GLANCE["chi_square_p_value_nb20"] = NB20_SUMMARY.get("chi_square_p_value")
if NB21_SUMMARY:
    AT_A_GLANCE["rank_ordering_pass_nb21"] = NB21_SUMMARY.get("rank_ordering_pass")
    AT_A_GLANCE["cramers_v"] = NB21_SUMMARY.get("cramers_v")
    AT_A_GLANCE["fair_lending_testing_status"] = NB21_SUMMARY.get("fair_lending_testing_status")
if NB22_SUMMARY:
    AT_A_GLANCE["api_self_test_passed"] = NB22_SUMMARY.get("api_self_test_passed")
    AT_A_GLANCE["api_latency_p50_ms"] = NB22_SUMMARY.get("api_latency_summary_ms", {}).get("p50_ms")
if NB23_SUMMARY:
    AT_A_GLANCE["monitoring_n_alerts"] = NB23_SUMMARY.get("n_alerts")
    AT_A_GLANCE["method_agreement_kappa"] = NB23_SUMMARY.get("method_agreement_kappa")

for _k, _v in AT_A_GLANCE.items():
    print(f"  {_k:<32}: {_v}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: BUSINESS & FINANCIAL IMPACT NARRATIVE (CONDENSED, HONEST SCOPE)
# =============================================================================
_section("SECTION 5: Business & Financial Impact Narrative (Condensed, Honest Scope)")

# --- Same honesty boundary Notebook 14 states for Problem 1: the Kaggle
#     dataset has no revenue, cost, or investment data for any real
#     institution. Every dollar figure below rests on an explicit, editable
#     ASSUMPTION, clearly separated from the real, measured tier statistics
#     that drive it (Notebook 20's real bad-rate-by-tier table). ---
if NB20_SUMMARY:
    _primary_method = NB20_SUMMARY["primary_method"]
    _tier_order_for_spread = RISK_TIER_POLICY["tier_order"]
    _bad_rate_by_tier = {r["risk_tier"]: r["actual_bad_rate_pct"] for r in NB20_SUMMARY["tier_performance"]
                          if r["method"] == _primary_method and r["actual_bad_rate_pct"] is not None}
    # --- Highest- vs. lowest-RISK tier specifically (by the policy's real
    #     tier_order), NOT max()/min() across all tiers -- a tiny-population
    #     "Prime" tier can show a noisy 100% bad rate by chance and must not
    #     be mistaken for the genuine high-risk end of the scale. ---
    _lowest_risk_tier = _tier_order_for_spread[0]
    _highest_risk_tier = _tier_order_for_spread[-1]
    if _lowest_risk_tier in _bad_rate_by_tier and _highest_risk_tier in _bad_rate_by_tier:
        REAL_BAD_RATE_SPREAD_PP = round(_bad_rate_by_tier[_highest_risk_tier] - _bad_rate_by_tier[_lowest_risk_tier], 3)
    else:
        REAL_BAD_RATE_SPREAD_PP = None
else:
    REAL_BAD_RATE_SPREAD_PP = None

SCENARIO_ASSUMPTIONS = {
    "deployment_portfolio_size_accounts": 500_000,          # ASSUMPTION -- illustrative mid-size card portfolio
    "avg_exposure_at_default_usd": 3_500,                    # ASSUMPTION -- illustrative average revolving balance
    "high_risk_tier_population_share": 0.10,                 # ASSUMPTION -- illustrative share of book in the top-risk tier
    "risk_based_repricing_loss_reduction_share": 0.15,       # ASSUMPTION -- illustrative share of high-risk-tier losses
                                                               # avoided via tier-differentiated pricing/limits/review
}
print("ASSUMPTION (stated, editable -- not measured from the Kaggle dataset):")
for k, v in SCENARIO_ASSUMPTIONS.items():
    print(f"  {k:<45}: {v:,}" if isinstance(v, int) else f"  {k:<45}: {v:.2%}")

if REAL_BAD_RATE_SPREAD_PP is not None:
    _high_risk_accounts = SCENARIO_ASSUMPTIONS["deployment_portfolio_size_accounts"] * SCENARIO_ASSUMPTIONS["high_risk_tier_population_share"]
    _high_risk_bad_rate = _bad_rate_by_tier[_highest_risk_tier] / 100.0
    _annual_high_risk_losses_usd = (_high_risk_accounts * _high_risk_bad_rate
                                     * SCENARIO_ASSUMPTIONS["avg_exposure_at_default_usd"])
    ANNUAL_ILLUSTRATIVE_LOSS_AVOIDANCE_USD = (_annual_high_risk_losses_usd
                                               * SCENARIO_ASSUMPTIONS["risk_based_repricing_loss_reduction_share"])
    print(f"\nReal, measured bad-rate spread ({_primary_method} method, Notebook 20): "
          f"{REAL_BAD_RATE_SPREAD_PP:.2f} percentage points (highest- vs. lowest-risk tier)")
    if REAL_BAD_RATE_SPREAD_PP < 0:
        print("  ⚠️  Negative spread -- the lowest-risk tier showed a HIGHER real bad rate than the "
              "highest-risk tier on this run. This is a genuine rank-ordering / monotonicity issue (Notebook 20's "
              "own KPI check reports it too), often driven by a very small tier population on a small holdout -- "
              "not corrected or hidden here.")
    print(f"Illustrative annual high-risk-tier exposure (ASSUMPTION portfolio): ${_annual_high_risk_losses_usd:,.0f}")
    print(f"Illustrative annual loss avoidance from risk-based tiering (ASSUMPTION): "
          f"${ANNUAL_ILLUSTRATIVE_LOSS_AVOIDANCE_USD:,.0f}")
else:
    ANNUAL_ILLUSTRATIVE_LOSS_AVOIDANCE_USD = None
    print("\nNotebook 20 has not been run yet -- financial narrative deferred until real tier bad rates exist.")

print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: CHARTS
# =============================================================================
_section("SECTION 6: Charts")

VIZ = {"surface": "#fcfcfb", "text_primary": "#0b0b0b", "text_secondary": "#52514e", "grid": "#e3e2dd",
       "cat_blue": "#2a78d6", "cat_green": "#3a9e5f", "cat_amber": "#d69a2a"}
PROBLEM_NAME = "Phase 1 \u00b7 Problem 2 -- Risk Tier Classification"


def _style_axes(ax):
    ax.set_facecolor(VIZ["surface"]); ax.figure.set_facecolor(VIZ["surface"])
    ax.grid(axis="y", color=VIZ["grid"], linewidth=0.8, zorder=0); ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(VIZ["grid"])
    ax.tick_params(colors=VIZ["text_secondary"], labelsize=9)
    ax.title.set_color(VIZ["text_primary"])


# Chart 1: notebook completion tracker
_status_colors = {"Complete": VIZ["cat_green"], "Not Yet Run": VIZ["cat_amber"]}
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
_colors = [_status_colors[s] for s in rollup_status_df["status"]]
ax.barh(rollup_status_df["notebook"].astype(str) + ": " + rollup_status_df["title"], [1] * len(rollup_status_df),
        color=_colors, zorder=3)
_style_axes(ax)
ax.set_xticks([])
ax.set_title(f"{PROBLEM_NAME}\nNotebook Completion Tracker (Real, This Session)", fontsize=11)
ax.invert_yaxis()
fig.tight_layout()
chart1_path = RISK_TIER_REPORTING_DIR / "notebook_completion_tracker_chart.png"
fig.savefig(chart1_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart1_path}")

_charts = [chart1_path]

if NB20_SUMMARY:
    _tier_perf_df = pd.DataFrame(NB20_SUMMARY["tier_performance"])
    _primary_perf = _tier_perf_df[_tier_perf_df["method"] == NB20_SUMMARY["primary_method"]]
    fig, ax = plt.subplots(figsize=(7.5, 5), dpi=150)
    _bars = ax.bar(_primary_perf["risk_tier"], _primary_perf["actual_bad_rate_pct"], color=VIZ["cat_blue"], zorder=3)
    ax.bar_label(_bars, padding=3, fontsize=9, fmt="%.1f%%")
    _style_axes(ax)
    ax.set_ylabel("Actual bad rate (%, real, Notebook 20)")
    ax.set_title(f"{PROBLEM_NAME}\nReal Bad Rate by Tier -- Primary Method ('{NB20_SUMMARY['primary_method']}')", fontsize=11)
    fig.tight_layout()
    chart2_path = RISK_TIER_REPORTING_DIR / "final_bad_rate_by_tier_chart.png"
    fig.savefig(chart2_path, dpi=150, facecolor=VIZ["surface"])
    plt.show(); plt.close(fig)
    print(f"\u2705 Saved -> {chart2_path}")
    _charts.append(chart2_path)

print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: SAVE ROLLUP ARTIFACT
# =============================================================================
_section("SECTION 7: Save Rollup Artifact")

risk_tier_rollup = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 2,
    "problem_name": "Risk Tier Classification",
    "notebook_status": _rollup_rows,
    "at_a_glance": AT_A_GLANCE,
    "scenario_assumptions": SCENARIO_ASSUMPTIONS,
    "real_bad_rate_spread_pp": REAL_BAD_RATE_SPREAD_PP,
    "annual_illustrative_loss_avoidance_usd": ANNUAL_ILLUSTRATIVE_LOSS_AVOIDANCE_USD,
}
rollup_path = RISK_TIER_REPORTING_DIR / "risk_tier_rollup.json"
with open(rollup_path, "w", encoding="utf-8") as f:
    json.dump(risk_tier_rollup, f, indent=2, default=str)
print(f"\u2705 Saved -> {rollup_path}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: WORD REPORT -- RISK_TIER_COMPREHENSIVE_REPORT.DOCX
# =============================================================================
_section("SECTION 8: Word Report -- Risk_Tier_Comprehensive_Report.docx")

doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 1, Problem 2: Risk Tier Classification -- Comprehensive Report")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

doc.add_heading("1. Notebook Completion Status", level=1)
_t = doc.add_table(rows=1, cols=len(rollup_status_df.columns))
_t.style = "Light Grid Accent 1"
for _i, _col in enumerate(rollup_status_df.columns):
    _t.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in rollup_status_df.iterrows():
    _cells = _t.add_row().cells
    for _i, _col in enumerate(rollup_status_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("2. At A Glance", level=1)
for _k, _v in AT_A_GLANCE.items():
    doc.add_paragraph(f"{_k.replace('_', ' ').title()}: {_v}")

doc.add_heading("3. Business & Financial Impact Narrative (Honest Scope)", level=1)
doc.add_paragraph(
    "This Kaggle dataset carries no revenue, cost, or investment data for any real institution -- every dollar "
    "figure below rests on an explicit, editable ASSUMPTION, separated from the real, measured tier statistics "
    "that drive it (Notebook 20's bad-rate-by-tier table)."
)
for k, v in SCENARIO_ASSUMPTIONS.items():
    doc.add_paragraph(f"ASSUMPTION -- {k.replace('_', ' ')}: {v}", style="List Bullet")
if ANNUAL_ILLUSTRATIVE_LOSS_AVOIDANCE_USD is not None:
    doc.add_paragraph(
        f"Real, measured bad-rate spread (highest- vs. lowest-risk tier): {REAL_BAD_RATE_SPREAD_PP:.2f} "
        f"percentage points. Illustrative annual loss avoidance from risk-based tiering, under the stated "
        f"assumptions: ${ANNUAL_ILLUSTRATIVE_LOSS_AVOIDANCE_USD:,.0f}."
    )
    if REAL_BAD_RATE_SPREAD_PP < 0:
        doc.add_paragraph(
            "Note: a negative spread means the lowest-risk tier showed a higher real bad rate than the "
            "highest-risk tier on this run -- a genuine rank-ordering issue, not corrected or hidden here "
            "(see Notebook 20's own KPI compliance report)."
        )
else:
    doc.add_paragraph("Notebook 20 has not been run yet -- financial narrative deferred.")

doc.add_heading("4. Charts", level=1)
for _cp in _charts:
    doc.add_picture(str(_cp), width=Inches(6.0))
    _p = doc.add_paragraph(_cp.stem.replace("_", " ").title()); _p.alignment = WD_ALIGN_PARAGRAPH.CENTER

report_path = RISK_TIER_REPORTING_DIR / "Risk_Tier_Comprehensive_Report.docx"
doc.save(report_path)
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 9: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("Rollup status table covers Notebooks 19-23", len(rollup_status_df) == 5, f"({len(rollup_status_df)})")
_check("Notebook 19 (required) reported Complete", rollup_status_df.loc[rollup_status_df["notebook"] == 19, "status"].iloc[0] == "Complete")
_check("AT_A_GLANCE has at least the 5 Notebook-19-sourced keys", len(AT_A_GLANCE) >= 5, f"({len(AT_A_GLANCE)})")

_expected_files = [rollup_path, report_path] + _charts
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 24 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 24 checks passed.")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 10: Resource / Performance Report")

_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "adaptive_ram_ceiling_gb": round(MAX_RAM_BYTES / 1e9, 2),
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
    "n_notebooks_complete": int(_n_complete),
}
performance_report_path = ARTIFACTS_DIR / "notebook_24_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)
print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)")
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: WRITE NOTEBOOK 24 SUMMARY ARTIFACT (for Notebook 25's packaging)
# =============================================================================
_section("SECTION 11: Write Notebook 24 Summary Artifact")

notebook_24_summary = {
    "notebook": "24_risk_tier_reporting",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 2,
    "problem_name": "Risk Tier Classification",
    "n_notebooks_complete": int(_n_complete),
    "at_a_glance": AT_A_GLANCE,
    "annual_illustrative_loss_avoidance_usd": ANNUAL_ILLUSTRATIVE_LOSS_AVOIDANCE_USD,
    "output_files": {p.name: str(p) for p in _expected_files + [performance_report_path]},
}
nb24_summary_path = ARTIFACTS_DIR / "notebook_24_summary.json"
with open(nb24_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_24_summary, f, indent=2, default=str)
print(f"\u2705 Saved -> {nb24_summary_path}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 12: Notebook 24 Complete -- Handoff to Notebook 25")

print("NOTEBOOK 24: RISK TIER COMPREHENSIVE REPORTING -- COMPLETE")
print(f"  Notebooks 19-23 complete           : {_n_complete}/5")
print(f"  Champion model                    : {AT_A_GLANCE['champion_model']}")
print(f"  Primary tiering method             : {AT_A_GLANCE['primary_method']}")
print(f"  Files produced                    : {len(_expected_files) + 2}")
for _p in _expected_files + [performance_report_path, nb24_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                     : 25_risk_tier_packaging.ipynb")
print("\n\u2705 Ready to proceed.")
